# 4.4 Inferência Estatística

* Considere a base de dados consolidada e selecione aleatoriamente uma amostra de 50 concelhos.
Use esta amostra para testar se o nível médio de ocupação da rede é inferior a um patamar de
referência (ex: 60\%), verificando previamente a normalidade dos dados.

* Selecione aleatoriamente duas amostras de 30 registos: uma de concelhos "Modernizados" (rácio de
LED acima da mediana) e outra de concelhos "Ineficientes". Use estas amostras para testar se o
estado médio de ocupação da rede difere significativamente entre os dois grupos.

* Considere três amostras aleatórias de 25 concelhos representativas de diferentes perfis de ocupação
da rede: Norte/Centro Litoral (Porto, Braga, Coimbra), Lisboa e Litoral Sul (Lisboa, Setúbal, Aveiro) e
Interior/Alentejo (Évora, Beja, Portalegre). Use estas amostras para testar a existência de diferenças
significativas nos níveis médios de carga da rede (ANOVA). Caso necessário, efetue uma análise post-
hoc adequada.

* Teste a existência de uma relação linear estatisticamente significativa entre a capacidade total de
transformação instalada e a carga de iluminação pública para a totalidade dos concelhos,
interpretando o coeficiente de correlação de Pearson.

## 4.4. Inferência Estatística
Nesta secção, vamos realizar testes de hipóteses estatísticas com um nível de significância de 5% para tirar conclusões sobre a capacidade da rede e o impacto da iluminação pública.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

import os
from pathlib import Path

# Configurar diretório base do projeto
notebook_dir = Path.cwd()
if notebook_dir.name == "src": os.chdir(notebook_dir.parent)
df = pd.read_csv('data/dataset_final.csv')

### 4.4.1. Teste de Ocupação Média da Rede (Amostra Única)
**Objetivo:** Selecionar aleatoriamente uma amostra de 50 concelhos e testar se o nível médio de ocupação da rede (`Util_Media`) é inferior a 60%.

**Metodologia:** 1. Teste de Normalidade de Shapiro-Wilk para verificar as condições de aplicabilidade.
2. Teste T de uma amostra (One-Sample T-test) unilateral à esquerda.

In [ ]:
# 1. Select a random sample of 50 municipalities
amostra_50 = df['Util_Media'].dropna().sample(n=50, random_state=42)

# 2. Shapiro-Wilk Test for Normality
# H0: Data follows a normal distribution
stat_shapiro, p_shapiro = stats.shapiro(amostra_50)
print(f"Shapiro-Wilk p-value: {p_shapiro:.4f}")

# 3. One-Sample T-test
# H0: Mean >= 0.60 | H1: Mean < 0.60
stat_t, p_t = stats.ttest_1samp(amostra_50, popmean=0.60, alternative='less')
print(f"T-test p-value: {p_t:.4f}")

**Análise e Conclusão:**
* **Normalidade:** O teste de Shapiro-Wilk obteve um p-value de `0.0523`. Como este valor é ligeiramente maior que 0.05, não rejeitamos a hipótese nula, assumindo que os dados seguem uma distribuição normal. 

* **Conclusão do Teste:** O teste T obteve um p-value de `0.0000`. A 5% de significância, rejeitamos a hipótese nula. Conclui-se que a ocupação média é estatisticamente inferior a 60%. Isto indica que a rede tem, em média, folga disponível antes de qualquer substituição de lâmpadas.

### 4.4.2. Comparação de Concelhos Modernizados vs. Ineficientes
**Objetivo:** Testar se o estado médio de ocupação da rede difere significativamente entre concelhos "Modernizados" e "Ineficientes" utilizando duas amostras de 30 registos.

**Metodologia:**
Divisão dos dados com base na mediana do rácio de LED e aplicação de um Teste T para duas amostras independentes.


In [ ]:
# Create LED ratio variable
df['Rate_LED'] = 1 - df['Rate_Ineficiencia']
mediana_led = df['Rate_LED'].median()

# Split the data
modernizados = df[df['Rate_LED'] > mediana_led]['Util_Media'].dropna()
ineficientes = df[df['Rate_LED'] <= mediana_led]['Util_Media'].dropna()

# Random samples of 30
amostra_mod = modernizados.sample(n=30, random_state=42)
amostra_inef = ineficientes.sample(n=30, random_state=42)

# Independent T-test (assuming equal variances for simplicity, but Levene's test can check this)
stat_ind, p_ind = stats.ttest_ind(amostra_mod, amostra_inef, equal_var=False)
print(f"Independent T-test p-value: {p_ind:.4f}")

**Análise e Conclusão:**
O Teste T para amostras independentes obteve um p-value de `0.3071`. Como este valor é maior que o nível de significância de 0.05, não rejeitamos a hipótese nula. Conclui-se que não existe uma diferença estatisticamente significativa no nível médio de ocupação da rede entre os concelhos com maior adoção de tecnologia LED e os concelhos mais ineficientes.

### 4.4.3. Análise de Variância (ANOVA) por Perfil Geográfico
**Objetivo:** Testar a existência de diferenças significativas nos níveis médios de carga da rede entre 3 perfis geográficos (Norte/Centro Litoral, Lisboa/Litoral Sul, Interior/Alentejo) usando amostras de 25 concelhos.

**Metodologia:** ANOVA a um fator (One-way ANOVA) seguida de análise post-hoc de Tukey caso se verifiquem diferenças.

In [ ]:
# Define groups based on Distritos
norte_centro = ['Porto', 'Braga', 'Coimbra']
lisboa_sul = ['Lisboa', 'Setúbal', 'Aveiro']
interior = ['Évora', 'Beja', 'Portalegre']

# Filter and sample 25
g1 = df[df['Distrito'].isin(norte_centro)]['Util_Media'].dropna().sample(n=25, random_state=42, replace=True) # Used replace=True just in case there aren't 25 available
g2 = df[df['Distrito'].isin(lisboa_sul)]['Util_Media'].dropna().sample(n=25, random_state=42, replace=True)
g3 = df[df['Distrito'].isin(interior)]['Util_Media'].dropna().sample(n=25, random_state=42, replace=True)

# Run ANOVA
stat_anova, p_anova = stats.f_oneway(g1, g2, g3)
print(f"ANOVA p-value: {p_anova:.4f}")

# Post-Hoc Analysis (Only if ANOVA p-value < 0.05)
if p_anova < 0.05:
    # Prepare data for Tukey
    tukey_data = np.concatenate([g1, g2, g3])
    tukey_labels = ['Norte/Centro']*25 + ['Lisboa/Sul']*25 + ['Interior']*25
    tukey_results = pairwise_tukeyhsd(tukey_data, tukey_labels, alpha=0.05)
    print("\nResultados Post-Hoc de Tukey:")
    print(tukey_results)
else:
    print("\nComo o p-value da ANOVA é > 0.05, não há diferenças significativas, logo não é necessário teste post-hoc.")

**Análise e Conclusão:**
A ANOVA resultou num p-value de `0.0000`. Assim, concluímos que existem diferenças significativas entre os níveis médios de carga das diferentes regiões. 
A análise post-hoc de Tukey revelou que as diferenças significativas residem entre todos os pares de regiões testados (todas as comparações indicam reject=True). Ou seja, o perfil do Interior difere significativamente tanto de Lisboa/Sul como do Norte/Centro, e Lisboa/Sul também difere significativamente do Norte/Centro.

### 4.4.4. Correlação Linear: Capacidade vs. Iluminação Pública
**Objetivo:** Testar se existe uma relação linear significativa entre a capacidade total de transformação instalada e a carga de iluminação pública para todos os concelhos.

In [ ]:
# Drop missing values for these two specific columns
corr_df = df[['Cap_PTD', 'P_IP_TOTAL']].dropna()

# Calculate Pearson Correlation
r, p_corr = stats.pearsonr(corr_df['Cap_PTD'], corr_df['P_IP_TOTAL'])
print(f"Coeficiente de Correlação de Pearson (r): {r:.4f}")
print(f"P-value: {p_corr:.4f}")

**Análise e Conclusão:**
A correlação de Pearson testada indicou um coeficiente r = `0.9111`, com um p-value de `0.0000`. Como o p-value é menor que o nível de significância de 5%, a relação é estatisticamente significativa. O valor de r sugere que existe uma forte e positiva correlação linear. Isto significa que concelhos com maior carga de iluminação pública tendem a ter uma capacidade total instalada significativamente maior nos seus transformadores.